In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
from m5forecast.reconciliation.hierarchy import (
    build_summing_matrix,
    reconcile_mint_shrink,
    LEVEL_ORDER,
    HIERARCHY
)

# 1. Load preprocessed bottom-level M5 data from Task 1
data_path = Path("data/processed/m5_subset.parquet") # Adjust path to your ingestion output if needed
df = pd.read_parquet(data_path) if data_path.exists() else None

print("Loading data and setting up hierarchy...")

# 2. Build Summing Matrix S using item and store identifiers
# Ensure your dataframe contains columns: item_id, store_id, dept_id, cat_id, state_id
id_cols = ['item_id', 'store_id']
# For demonstration/mocking if running live against your ingestion table:
# bottom_df = df.drop_duplicates(subset=id_cols)
# S, all_ids = build_summing_matrix(bottom_df, id_cols=id_cols)

# 3. Helper to map series IDs to hierarchy levels for evaluation
def get_level_mask(all_ids: list[str], level_name: str) -> np.ndarray:
    """Returns a boolean mask for series belonging to a specific hierarchy level."""
    # Logic matching your HIERARCHY mapping prefixes/suffixes
    # ...
    pass

print("Ready to compute MinT reconciliation and evaluate WAPE across levels!")

Loading data and setting up hierarchy...
Ready to compute MinT reconciliation and evaluate WAPE across levels!


In [2]:
import numpy as np

# 1. Define the number of bottom-level series per store (adjust this if you have a different number of items)
items_per_store = 10  # For example, if each store has 10 bottom-level series
n_stores = 3          # CA_1, TX_1, WI_1
n_bottom = n_stores * items_per_store  # Total bottom-level series = 30

# 2. Build the hierarchical summing blocks:
# - Total level: sums all bottom series together (1 row of all 1s)
S_total = np.ones((1, n_bottom))

# - Store level: blocks of 1s for each respective store
S_store = np.kron(np.eye(n_stores), np.ones((1, items_per_store)))

# - Bottom level: identity matrix (each bottom series maps to itself)
S_bottom = np.eye(n_bottom)

# 3. Stack them vertically to form the complete summing matrix S
S = np.vstack([S_total, S_store, S_bottom])

# 4. Define LEVEL_ORDER to match your aggregation hierarchy layout
LEVEL_ORDER = ['total', 'store', 'bottom']

print(f"Successfully defined S with shape: {S.shape}")

Successfully defined S with shape: (34, 30)


In [8]:
import numpy as np
import pandas as pd

# 1. Define Hierarchical Structure & Summing Matrix (S)
items_per_store = 10  # e.g., 10 bottom-level series per store
n_stores = 3          # CA_1, TX_1, WI_1
n_bottom = n_stores * items_per_store  # Total bottom-level series = 30

S_total = np.ones((1, n_bottom))
S_store = np.kron(np.eye(n_stores), np.ones((1, items_per_store)))
S_bottom = np.eye(n_bottom)

S = np.vstack([S_total, S_store, S_bottom])
LEVEL_ORDER = ['total', 'store', 'bottom']

# 2. Setup Data Structures & Simulation
horizon = 28  # standard forecast horizon
np.random.seed(42)

actuals_bottom = np.random.uniform(10, 100, size=(n_bottom, horizon))
base_forecasts_bottom = actuals_bottom * np.random.uniform(0.9, 1.1, size=(n_bottom, horizon))
residuals_bottom = np.random.normal(0, 5, size=(n_bottom, 60)) # 60 days of historical residuals

# Map to full hierarchy using S
actuals_all = S @ actuals_bottom
base_forecasts_all = S @ base_forecasts_bottom
residuals_all = S @ residuals_bottom

# 3. Define MinT Shrinkage Reconciliation Function
def reconcile_mint_shrink(base_forecasts, S, residuals):
    W = np.cov(residuals)
    inv_W = np.linalg.pinv(W)
    St_invW_S = np.linalg.pinv(S.T @ inv_W @ S)
    P = S @ St_invW_S @ S.T @ inv_W
    reconciled = P @ base_forecasts
    return reconciled

# 4. Apply MinT Shrinkage Reconciliation
reconciled_mint = reconcile_mint_shrink(
    base_forecasts=base_forecasts_all,
    S=S,
    residuals=residuals_all
)

# 5. Define WAPE Metric Function
def compute_wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

# 6. Evaluate and Compare Base vs MinT WAPE Across Levels
print(f"{'Level':<15} | {'Base WAPE':<12} | {'MinT WAPE':<12} | {'Improvement':<12}")
print("-" * 60)

start_idx = 0
for level_name in LEVEL_ORDER:
    if level_name == "total":
        end_idx = start_idx + 1
    elif level_name == "store":
        end_idx = start_idx + n_stores
    else:
        end_idx = start_idx + n_bottom

    y_true_level = actuals_all[start_idx:end_idx]
    y_base_level = base_forecasts_all[start_idx:end_idx]
    y_mint_level = reconciled_mint[start_idx:end_idx]

    b_wape = compute_wape(y_true_level, y_base_level)
    m_wape = compute_wape(y_true_level, y_mint_level)
    improvement = ((b_wape - m_wape) / b_wape) * 100 if b_wape > 0 else 0.0

    print(f"{level_name:<15} | {b_wape:.4f}       | {m_wape:.4f}       | {improvement:+.1f}%")
    start_idx = end_idx

Level           | Base WAPE    | MinT WAPE    | Improvement 
------------------------------------------------------------
total           | 0.0110       | 0.0110       | +0.0%
store           | 0.0170       | 0.0170       | -0.0%
bottom          | 0.0510       | 0.0510       | +0.0%
